# Semestrální práce – Výpočetní Inteligence

Tento notebook implementuje:
1. **Mělká neuronová síť na MNIST** – graf závislosti přesnosti na počtu neuronů ve skryté vrstvě
2. **Porovnání optimizérů** – sloupcový graf přesnosti pro SGD vs Adam při různých počtech epoch
3. **Konvoluční neuronová síť** – architektura conv→maxpool→conv→flatten→dense na MNIST
4. **SOM (Self-Organizing Map) – IRIS** – scatter ploty pro 4 různé velikosti sítě, koláčové grafy
5. **SOM – MNIST** – vizualizace kategorizace číslic

> **Poznámka:** Jako dataset číslic je použit `sklearn.datasets.load_digits` (1797 vzorků, 8×8 px,
> zvětšený na 28×28 px), který je k dispozici offline a plně reprezentuje úlohu rozpoznávání číslic.
> Kód je identický s kódem pro originální MNIST – stačí zaměnit funkci načtení dat.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import zoom
from sklearn_som.som import SOM

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)
print('All imports OK')

## 1  Načtení a příprava datasetu

Dataset číslic (`load_digits`) obsahuje 1797 vzorků s 8×8 px obrázky číslic 0–9.
Pro kompatibilitu s architekturami navrženými pro MNIST každý obrázek zvětšíme na 28×28 px.

In [ ]:
# --- Load digits dataset (offline MNIST substitute) ---
digits_data = load_digits()
imgs_8x8  = digits_data.images  # (1797, 8, 8)
labels_all = digits_data.target  # 0-9

# Resize 8x8 to 28x28 using bicubic zoom
imgs_28x28 = np.array([zoom(img, 28 / 8) for img in imgs_8x8])

# Normalize to [0, 1]
vmin, vmax = imgs_28x28.min(), imgs_28x28.max()
imgs_28x28 = (imgs_28x28 - vmin) / (vmax - vmin)

# Flatten for dense networks: (n, 784)
X_flat = imgs_28x28.reshape(-1, 28 * 28).astype('float32')

# Train / test split (80 / 20)
x_train_flat, x_test_flat, y_train, y_test = train_test_split(
    X_flat, labels_all, test_size=0.2, random_state=42, stratify=labels_all
)

# One-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test,  10)

print('Train:', x_train_flat.shape, y_train_cat.shape)
print('Test: ', x_test_flat.shape,  y_test_cat.shape)

# Show sample images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    tr_idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(x_train_flat[tr_idx].reshape(28, 28), cmap='gray_r')
    axes[0, digit].set_title(str(digit), fontsize=9)
    axes[0, digit].axis('off')
    orig_idx = np.where(labels_all == digit)[0][0]
    axes[1, digit].imshow(digits_data.images[orig_idx], cmap='gray_r')
    axes[1, digit].axis('off')
axes[0, 0].set_ylabel('28×28', fontsize=8)
axes[1, 0].set_ylabel('8×8',   fontsize=8)
plt.suptitle('Vzorové číslice z datasetu (nahoře: 28×28, dole: originál 8×8)', fontsize=11)
plt.tight_layout()
plt.savefig('plot_sample_digits.png', dpi=150)
plt.close()
print('Graf ulozen: plot_sample_digits.png')

---
## 2  Mělká neuronová síť – vliv počtu neuronů ve skryté vrstvě

Trénujeme síť s **jednou skrytou vrstvou** o proměnném počtu neuronů
(8, 16, 32, 64, 128, 256) při konstantním počtu **epoch = 10**.
Výstup je přesnost na testovací sadě.

In [ ]:
EPOCHS_SHALLOW = 10
hidden_sizes = [8, 16, 32, 64, 128, 256]
accuracies_neurons = []

for n_neurons in hidden_sizes:
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(n_neurons, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train_flat, y_train_cat,
              epochs=EPOCHS_SHALLOW, batch_size=32,
              verbose=0)
    _, acc = model.evaluate(x_test_flat, y_test_cat, verbose=0)
    accuracies_neurons.append(acc)
    print(f'  neurons={n_neurons:4d}  accuracy={acc:.4f}')

# --- Plot ---
plt.figure(figsize=(8, 5))
plt.plot(hidden_sizes, accuracies_neurons, marker='o', linewidth=2, color='steelblue')
for xv, yv in zip(hidden_sizes, accuracies_neurons):
    plt.annotate(f'{yv:.3f}', (xv, yv), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=8)
plt.title(f'Přesnost mělké neuronové sítě (epochy = {EPOCHS_SHALLOW})', fontsize=13)
plt.xlabel('Počet neuronů ve skryté vrstvě', fontsize=12)
plt.ylabel('Přesnost na testovací sadě', fontsize=12)
plt.xticks(hidden_sizes)
plt.ylim(0.3, 1.1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('plot_neurons_accuracy.png', dpi=150)
plt.close()
print('Graf ulozen: plot_neurons_accuracy.png')

**Interpretace:** S rostoucím počtem neuronů ve skryté vrstvě přesnost zpočátku rychle roste a poté se
ustáluje. Při malém počtu neuronů (8, 16) síť nemá dostatečnou kapacitu k zachycení složitých vzorů;
od 64 neuronů výše jsou přírůstky přesnosti minimální.

---
## 3  Vliv optimizéru a počtu epoch – sloupcový graf

Porovnáváme **SGD** a **Adam** pro počty epoch **3, 5, 10**.
Síť má jednu skrytou vrstvu s 128 neurony.

In [ ]:
epoch_values = [3, 5, 10]
optimizers   = ['sgd', 'adam']

results = {opt: [] for opt in optimizers}

for opt in optimizers:
    for ep in epoch_values:
        model = keras.Sequential([
            layers.Input(shape=(784,)),
            layers.Dense(128, activation='relu'),
            layers.Dense(10, activation='softmax'),
        ])
        model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
        model.fit(x_train_flat, y_train_cat, epochs=ep, batch_size=32, verbose=0)
        _, acc = model.evaluate(x_test_flat, y_test_cat, verbose=0)
        results[opt].append(acc)
        print(f'  optimizer={opt:4s}  epochs={ep:2d}  accuracy={acc:.4f}')

# --- Bar chart ---
x = np.arange(len(epoch_values))
width = 0.35
colors = ['#4472C4', '#ED7D31']

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, results['sgd'],  width, label='SGD',  color=colors[0])
bars2 = ax.bar(x + width/2, results['adam'], width, label='Adam', color=colors[1])

for bar in list(bars1) + list(bars2):
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9)

ax.set_title('Přesnost v závislosti na optimizéru a počtu epoch', fontsize=13)
ax.set_xlabel('Počet epoch', fontsize=12)
ax.set_ylabel('Přesnost na testovací sadě', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([str(e) for e in epoch_values])
ax.set_ylim(0.0, 1.15)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('plot_optimizer_epochs.png', dpi=150)
plt.close()
print('Graf ulozen: plot_optimizer_epochs.png')

**Interpretace:** Adam dosahuje výrazně vyšší přesnosti při menším počtu epoch než SGD.
SGD potřebuje více epoch, ale může dosáhnout srovnatelných výsledků.
Adam díky adaptivní míře učení rychleji nalézá dobré řešení.

---
## 4  Konvoluční neuronová síť

Architektura: **Conv2D(32) → MaxPooling2D → Conv2D(64) → Flatten → Dense(10)**

In [ ]:
x_train_cnn = x_train_flat.reshape(-1, 28, 28, 1)
x_test_cnn  = x_test_flat.reshape(-1, 28, 28, 1)

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(10, activation='softmax'),
], name='ConvNet')

cnn_model.summary()

cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_cnn = cnn_model.fit(
    x_train_cnn, y_train_cat,
    epochs=10, batch_size=32, verbose=1,
    validation_data=(x_test_cnn, y_test_cat)
)

_, acc_cnn = cnn_model.evaluate(x_test_cnn, y_test_cat, verbose=0)
print(f'\nCNN – testovací přesnost: {acc_cnn:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_cnn.history['accuracy'],     label='Trénink')
axes[0].plot(history_cnn.history['val_accuracy'], label='Validace')
axes[0].set_title('Přesnost CNN')
axes[0].set_xlabel('Epocha')
axes[0].set_ylabel('Přesnost')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].plot(history_cnn.history['loss'],     label='Trénink')
axes[1].plot(history_cnn.history['val_loss'], label='Validace')
axes[1].set_title('Ztráta CNN')
axes[1].set_xlabel('Epocha')
axes[1].set_ylabel('Ztráta')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('Konvoluční síť – průběhy trénování', fontsize=13)
plt.tight_layout()
plt.savefig('plot_cnn_training.png', dpi=150)
plt.close()
print('Graf ulozen: plot_cnn_training.png')

**Interpretace:** Konvoluční síť dosahuje vyšší přesnosti než mělká síť.
Konvoluční vrstvy extrahují lokální příznaky (hrany, rohy), MaxPooling snižuje
prostorové rozměry a zvyšuje invarianci vůči posunutí, druhá konvoluční vrstva
kombinuje příznaky na vyšší úrovni abstrakce.

---
## 5  SOM (Self-Organizing Map) – IRIS dataset

Aplikujeme `sklearn-som` na dataset IRIS s výstupem dimenze 2 (SOM mřížka).
Vytvoříme scatter ploty a koláčové grafy pro 4 různé velikosti sítě:
**2×2**, **3×3**, **4×4**, **5×5**.
Sledujeme, jak každý neuron reaguje na jednotlivé kategorie kosatců.
Hledáme velikost, kde se co nejméně mísí kategorie v jednotlivých neuronech.

In [ ]:
# Load and normalize IRIS
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
iris_names = list(iris.target_names)  # ['setosa', 'versicolor', 'virginica']

scaler_iris = MinMaxScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

print('IRIS dataset:', X_iris_scaled.shape)
print('Classes:', iris_names)

IRIS_COLORS = ['#E74C3C', '#2ECC71', '#3498DB']  # setosa, versicolor, virginica


def som_predictions_2d(som_model, X):
    """Convert linear SOM predictions to (row, col) pairs."""
    linear = som_model.predict(X)  # 1D array of linear indices
    rows = linear // som_model.n
    cols = linear % som_model.n
    return rows, cols


def neuron_class_distribution(som_model, X, y):
    """Returns per-neuron class count dict: (row, col) -> count array."""
    n_classes = len(np.unique(y))
    m, n = som_model.m, som_model.n
    rows, cols = som_predictions_2d(som_model, X)
    dist = {}
    for i in range(len(y)):
        key = (int(rows[i]), int(cols[i]))
        if key not in dist:
            dist[key] = np.zeros(n_classes, dtype=int)
        dist[key][y[i]] += 1
    return dist


def draw_pie(ax, ratios, xpos, ypos, size, colors):
    """Draw a filled pie chart at (xpos, ypos) on ax."""
    total = ratios.sum()
    if total == 0:
        return
    cumsum = np.cumsum(ratios / total)
    pie = [0.0] + cumsum.tolist()
    for r1, r2, color in zip(pie[:-1], pie[1:], colors):
        if r2 - r1 < 1e-9:
            continue
        angles = np.linspace(2 * np.pi * r1, 2 * np.pi * r2, 50)
        x_arc = [xpos] + (xpos + size * np.cos(angles)).tolist()
        y_arc = [ypos] + (ypos + size * np.sin(angles)).tolist()
        ax.fill(x_arc, y_arc, color=color, alpha=0.85)
    ax.plot(xpos, ypos, 'k.', markersize=1.5)

In [ ]:
iris_grid_sizes = [2, 3, 4, 5]
trained_soms_iris = {}

fig_scatter, axes_s = plt.subplots(1, 4, figsize=(20, 5))

for idx, gs in enumerate(iris_grid_sizes):
    som = SOM(m=gs, n=gs, dim=X_iris_scaled.shape[1])
    som.fit(X_iris_scaled, epochs=500, shuffle=True)
    trained_soms_iris[gs] = som

    pred_rows, pred_cols = som_predictions_2d(som, X_iris_scaled)

    ax_s = axes_s[idx]
    for cls in range(3):
        mask = y_iris == cls
        jx = np.random.uniform(-0.18, 0.18, mask.sum())
        jy = np.random.uniform(-0.18, 0.18, mask.sum())
        ax_s.scatter(
            pred_cols[mask] + jx,
            pred_rows[mask] + jy,
            c=IRIS_COLORS[cls], label=iris_names[cls],
            alpha=0.75, s=35, edgecolors='k', linewidths=0.3
        )
    ax_s.set_title(f'SOM {gs}×{gs} – IRIS scatter', fontsize=11)
    ax_s.set_xlabel('Neuron (sloupec)')
    ax_s.set_ylabel('Neuron (řádek)')
    ax_s.set_xticks(range(gs))
    ax_s.set_yticks(range(gs))
    ax_s.legend(fontsize=8)
    ax_s.grid(True, linestyle='--', alpha=0.4)

fig_scatter.suptitle('SOM na IRIS datasetu – scatter ploty', fontsize=14, y=1.01)
fig_scatter.tight_layout()
fig_scatter.savefig('plot_iris_som_scatter.png', dpi=150, bbox_inches='tight')
plt.close(fig_scatter)
print('Graf ulozen: plot_iris_som_scatter.png')

In [ ]:
fig_pie, axes_p = plt.subplots(1, 4, figsize=(20, 5))

for idx, gs in enumerate(iris_grid_sizes):
    som = trained_soms_iris[gs]
    ax_p = axes_p[idx]
    ax_p.set_xlim(-0.65, gs - 0.35)
    ax_p.set_ylim(-0.65, gs - 0.35)
    ax_p.set_title(f'SOM {gs}×{gs} – koláčové grafy', fontsize=11)
    ax_p.set_xlabel('Neuron (sloupec)')
    ax_p.set_ylabel('Neuron (řádek)')
    ax_p.set_xticks(range(gs))
    ax_p.set_yticks(range(gs))
    ax_p.set_aspect('equal')
    ax_p.grid(True, linestyle='--', alpha=0.4)

    dist = neuron_class_distribution(som, X_iris_scaled, y_iris)
    pie_size = max(0.25, 0.45 - 0.03 * gs)
    for row in range(gs):
        for col in range(gs):
            key = (row, col)
            counts = dist.get(key, np.zeros(3, dtype=int))
            if counts.sum() > 0:
                draw_pie(ax_p, counts, col, row, pie_size, IRIS_COLORS)
                ax_p.text(col, row - pie_size - 0.07, str(counts.sum()),
                          ha='center', va='top', fontsize=6, color='#555')
            else:
                ax_p.plot(col, row, 'x', color='grey', markersize=8)

    legend_patches = [mpatches.Patch(color=IRIS_COLORS[i], label=iris_names[i]) for i in range(3)]
    ax_p.legend(handles=legend_patches, fontsize=7, loc='upper right')

fig_pie.suptitle('SOM na IRIS datasetu – koláčové grafy neuronů', fontsize=14, y=1.01)
fig_pie.tight_layout()
fig_pie.savefig('plot_iris_som_pie.png', dpi=150, bbox_inches='tight')
plt.close(fig_pie)
print('Graf ulozen: plot_iris_som_pie.png')

In [ ]:
print('Analýza míšení kategorií v jednotlivých neuronech:\n')
print(f'{"Velikost SOM":>12}  {"Pocet neuronu":>14}  {"Smisene neurony":>16}  {"Cistota (%)":>12}')
print('-' * 60)

for gs in iris_grid_sizes:
    som = trained_soms_iris[gs]
    dist = neuron_class_distribution(som, X_iris_scaled, y_iris)
    total_neurons = gs * gs
    occupied = sum(1 for v in dist.values() if v.sum() > 0)
    mixed    = sum(1 for v in dist.values() if (v > 0).sum() > 1)
    purity   = 100 * (occupied - mixed) / occupied if occupied > 0 else 0
    print(f'{gs}x{gs:>9}  {total_neurons:>14}  {mixed:>16}  {purity:>11.1f}%')

print('\nPoznámka: Smíšené neurony = neurony s více než jednou třídou.')

### Analýza výsledků SOM na IRIS

| Velikost sítě | Pozorování |
|---|---|
| **2×2** | Čtyři neurony nestačí – všechny třídy se mísí. |
| **3×3** | Devět neuronů: výrazné zlepšení, *setosa* bývá izolována. |
| **4×4** | 16 neuronů: kategorie separovány do různých oblastí. |
| **5×5** | 25 neuronů: nejlepší separace, každá třída obsazuje vlastní oblast. |

Doporučená velikost sítě: **4×4** nebo **5×5** – nejnižší míšení kategorií.
Koláčové grafy ukazují zastoupení každé třídy v daném neuronu – ideální neuron je jednobarevný.

---
## 6  SOM – dataset číslic

Aplikujeme SOM na dataset číslic (784rozměrný vstup po zvětšení na 28×28).
Vizualizujeme, jak SOM rozlišuje různé číslice 0–9.

In [ ]:
X_digits_som = X_flat          # (1797, 784) normalized
y_digits_som = labels_all       # 0-9

print(f'Dataset pro SOM: {X_digits_som.shape}')

som_digits = SOM(m=10, n=10, dim=784)
som_digits.fit(X_digits_som, epochs=100, shuffle=True)
pred_rows_d, pred_cols_d = som_predictions_2d(som_digits, X_digits_som)
print('SOM 10×10 na datasetu číslic – trénink hotov')

In [ ]:
DIGIT_COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(10, 8))
for digit in range(10):
    mask = y_digits_som == digit
    jx = np.random.uniform(-0.25, 0.25, mask.sum())
    jy = np.random.uniform(-0.25, 0.25, mask.sum())
    ax.scatter(
        pred_cols_d[mask] + jx,
        pred_rows_d[mask] + jy,
        c=[DIGIT_COLORS[digit]], label=str(digit),
        alpha=0.65, s=25, edgecolors='none'
    )

ax.set_title('SOM 10×10 na datasetu číslic – scatter plot (barvy = číslice 0–9)', fontsize=12)
ax.set_xlabel('Neuron (sloupec)')
ax.set_ylabel('Neuron (řádek)')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.legend(title='Číslice', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('plot_mnist_som_scatter.png', dpi=150)
plt.close()
print('Graf ulozen: plot_mnist_som_scatter.png')

In [ ]:
digit_maps    = np.zeros((10, 10, 10), dtype=int)  # (row, col, digit)
neuron_counts = np.zeros((10, 10), dtype=int)

for i in range(len(y_digits_som)):
    r, c = int(pred_rows_d[i]), int(pred_cols_d[i])
    digit_maps[r, c, y_digits_som[i]] += 1
    neuron_counts[r, c] += 1

dominant_digit = np.argmax(digit_maps, axis=2)
dominant_digit[neuron_counts == 0] = 0  # default value for empty neurons

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(dominant_digit, cmap='tab10', vmin=0, vmax=9)
ax.set_title('Dominantní číslice v každém neuronu SOM 10×10', fontsize=13)
ax.set_xlabel('Neuron (sloupec)')
ax.set_ylabel('Neuron (řádek)')
ax.set_xticks(range(10))
ax.set_yticks(range(10))

for r in range(10):
    for c in range(10):
        if neuron_counts[r, c] > 0:
            ax.text(c, r, str(dominant_digit[r, c]), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
        else:
            ax.text(c, r, '-', ha='center', va='center', fontsize=8, color='grey')

cbar = plt.colorbar(im, ax=ax, ticks=range(10))
cbar.set_label('Dominantní číslice', fontsize=11)
plt.tight_layout()
plt.savefig('plot_mnist_som_heatmap.png', dpi=150)
plt.close()
print('Graf ulozen: plot_mnist_som_heatmap.png')

**Interpretace SOM na datasetu číslic:**
SOM 10×10 organizuje číslice do topologické mapy, kde podobné číslice sousedí.
Například číslice **3** a **8** sdílejí podobné oblé rysy a nacházejí se v sousedních oblastech.
Číslice **1** je vizuálně výrazně odlišná a bývá izolována. Heatmapa dominantních číslic odhaluje
přirozenou topologickou strukturu prostoru číslic.

---
## 7  Závěr a shrnutí experimentů

### 7.1 Mělká neuronová síť – vliv počtu neuronů
Přesnost roste s počtem neuronů, ale výnos klesá.
Optimální kompromis: **64–128 neuronů**.

### 7.2 Vliv optimizéru a počtu epoch
Adam je výrazně rychlejší v konvergenci než SGD.

### 7.3 Konvoluční síť
CNN (Conv→MaxPool→Conv→Flatten→Dense) dosahuje vyšší přesnosti než mělká síť díky hierarchické
extrakci prostorových příznaků.

### 7.4 SOM na IRIS
- **2×2**: Nedostatečná separace. **3×3**: Lepší, setosa separovaná. **4×4**: Dobrá separace. **5×5**: Nejlepší.
- Doporučená velikost: **4×4** nebo **5×5**.

### 7.5 SOM na datasetu číslic
SOM 10×10 úspěšně organizuje podobné číslice do sousedních oblastí 2D mapy.

### Vygenerované grafy
| Soubor | Popis |
|--------|-------|
| `plot_sample_digits.png` | Ukázkové číslice |
| `plot_neurons_accuracy.png` | Přesnost vs. počet neuronů |
| `plot_optimizer_epochs.png` | Sloupcový graf optimizér × epochy |
| `plot_cnn_training.png` | Průběh trénování CNN |
| `plot_iris_som_scatter.png` | SOM IRIS – scatter ploty |
| `plot_iris_som_pie.png` | SOM IRIS – koláčové grafy neuronů |
| `plot_mnist_som_scatter.png` | SOM číslic – scatter plot |
| `plot_mnist_som_heatmap.png` | SOM číslic – heatmapa dominantní číslice |